In [ ]:
"""
Fire-LLaVA: FINAL FIX (Expanded Token Limit)
============================================
1. Fixes 'Mismatch' error by increasing max_length to 2048.
2. Keeps the auto-repair and formatting fixes.
3. GUARANTEED to fit the image tokens.
"""

import os
import glob
import torch
import jsonlines
import shutil
from PIL import Image
from google.colab import drive
import warnings
from transformers import (
    AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig,
    TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset

# 1. SETUP
print(" Initializing Final Fix...")
warnings.filterwarnings('ignore')
drive.mount('/content/drive', force_remount=True)

class Config:
    DRIVE_ROOT = "/content/drive/MyDrive/ScenesolverG338LLaVAcoreLLM"
    FRAMES_ROOT = f"{DRIVE_ROOT}/FireLLaVA_frames"
    MANIFEST_FILE = f"{DRIVE_ROOT}/FireLLaVA_manifests/train_manifest.jsonl"
    OUTPUT_DIR = f"{DRIVE_ROOT}/FireLLaVA_checkpoints"
    MODEL_ID = "llava-hf/llava-1.5-7b-hf"

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

# ============================================================================
# SECTION 2: ENSURE MANIFEST EXISTS
# ============================================================================
# We don't delete it this time if it exists, because we just built a good one.
# But we double check it has data.

if not os.path.exists(config.MANIFEST_FILE):
    print(" Building manifest...")
    images = glob.glob(f"{config.FRAMES_ROOT}/**/*.jpg", recursive=True)
    with jsonlines.open(config.MANIFEST_FILE, 'w') as writer:
        for img in images:
            writer.write({
                "id": os.path.basename(img),
                "image": img,
                "conversations": [
                    {
                        "role": "user",
                        "content": [{"type": "image"}, {"type": "text", "text": "Describe the anomaly."}]
                    },
                    {
                        "role": "assistant",
                        "content": [{"type": "text", "text": "Anomaly detected."}]
                    }
                ]
            })

# ============================================================================
# SECTION 3: MODEL SETUP
# ============================================================================

print(" Loading Model...")
processor = AutoProcessor.from_pretrained(config.MODEL_ID)
processor.patch_size = 14
processor.vision_feature_select_strategy = "default"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)

model = LlavaForConditionalGeneration.from_pretrained(
    config.MODEL_ID, quantization_config=bnb_config, device_map="auto"
)

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# ============================================================================
# SECTION 4: DATASET (THE FIX IS HERE)
# ============================================================================

class FinalDataset(Dataset):
    def __init__(self, manifest_path, processor):
        self.data = []
        with jsonlines.open(manifest_path) as reader:
            for obj in reader: self.data.append(obj)
        self.processor = processor

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item['image']).convert('RGB')

        formatted_prompt = processor.apply_chat_template(
            item['conversations'],
            add_generation_prompt=False
        )

        # --- THE FIX ---
        # Changed max_length from 128 -> 2048
        # LLaVA needs 576 tokens just for the image. 2048 is safe.
        inputs = self.processor(
            images=image,
            text=formatted_prompt,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=2048
        )

        return {
            "input_ids": inputs["input_ids"][0],
            "attention_mask": inputs["attention_mask"][0],
            "pixel_values": inputs["pixel_values"][0],
            "labels": inputs["input_ids"][0]
        }

dataset = FinalDataset(config.MANIFEST_FILE, processor)

args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_steps=20, # Short run to verify
    learning_rate=2e-4,
    logging_steps=1,
    fp16=True,
    report_to="none"
)

trainer = Trainer(model=model, args=args, train_dataset=dataset)

print(" Starting Training (2048 token limit)...")
trainer.train()
trainer.save_model(f"{config.OUTPUT_DIR}/final_adapter")
print(" DONE. You can relax now.")

📦 Initializing Final Fix...
Mounted at /content/drive
🤖 Loading Model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

🚀 Starting Training (2048 token limit)...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,13.084600
2,12.914900
3,12.709500
4,12.323600
5,11.975500
6,11.637700
7,11.380800
8,11.121600
9,10.933800
10,10.718200


✅ DONE. You can relax now.


In [ ]:
"""
Fire-LLaVA: The REAL Training Run (1 Full Epoch)
================================================
1. Uses the exact same "Safe" configuration that just worked.
2. Trains for 1 Full Epoch (approx 62 steps).
3. Should bring Loss down significantly (aiming for < 5.0).
"""

from transformers import TrainingArguments, Trainer
import torch

# We reuse the 'model', 'dataset', 'processor', and 'data_collator'
# from the previous cell. No need to reload them!

print(" Configuring for Full Training Run...")

# Calculate steps: 500 images / 8 (batch size) = ~62 steps per epoch
# We will run 1 epoch.

args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    per_device_train_batch_size=1,     # Keep low for safety
    gradient_accumulation_steps=8,     # Aggregates to effective batch of 8
    num_train_epochs=1,                # 1 Full pass through all data
    learning_rate=2e-4,
    logging_steps=5,                   # Log every 5 steps so we see progress
    fp16=True,
    save_strategy="steps",
    save_steps=20,                     # Save backup every 20 steps
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(model=model, args=args, train_dataset=dataset)

print(f" Starting 1-Epoch Training Run...")
print("This should take about 30-45 minutes. Watch the 'Loss' number go down!")

trainer.train()

print(" Saving Final Model...")
trainer.save_model(f"{config.OUTPUT_DIR}/final_firellava_v1")
print(" DONE. Now we can test.")

🚀 Configuring for Full Training Run...
🔥 Starting 1-Epoch Training Run...
This should take about 30-45 minutes. Watch the 'Loss' number go down!


Step,Training Loss
5,9.932600
10,9.829000
15,9.784100
20,9.751300
25,9.727400
30,9.715400
35,9.705700
40,9.696800
45,9.689600
50,9.684000


✅ Saving Final Model...
🎉 DONE. Now we can test.


In [ ]:
"""
STEP 1: Smart Manifest Builder
==============================
Connects images to JSON descriptions by checking Folder Names.
"""
import os
import glob
import json
import random
import jsonlines

# PATHS
DRIVE_ROOT = "/content/drive/MyDrive/ScenesolverG338LLaVAcoreLLM"
ANNOT_ROOT = f"{DRIVE_ROOT}/Surveillance-Video-Understanding/UCF_Annotation"
JSON_DIR = f"{ANNOT_ROOT}/json"
FRAMES_ROOT = f"{DRIVE_ROOT}/FireLLaVA_frames"
MANIFEST_DIR = f"{DRIVE_ROOT}/FireLLaVA_manifests"

os.makedirs(MANIFEST_DIR, exist_ok=True)

# 1. LOAD DESCRIPTIONS
print("[STATUS] Loading JSON Data...")
def load_json(path):
    if not os.path.exists(path): return {}
    with open(path, 'r') as f: return json.load(f)

train_json = load_json(f"{JSON_DIR}/UCFCrime_Train.json")
val_json = load_json(f"{JSON_DIR}/UCFCrime_Val.json")
test_json = load_json(f"{JSON_DIR}/UCFCrime_Test.json")

# Create a lookup dictionary: "Abuse001_x264" -> ["Description 1", "Description 2"]
video_lookup = {}
for data_source in [train_json, val_json, test_json]:
    for vid_id, content in data_source.items():
        if "sentences" in content:
            video_lookup[vid_id] = content["sentences"]

print(f"[INFO] Loaded descriptions for {len(video_lookup)} videos.")

# 2. SCAN IMAGES & MATCH
print("[STATUS] Scanning Images and Matching...")
all_images = glob.glob(f"{FRAMES_ROOT}/**/*.jpg", recursive=True)

train_entries, val_entries, test_entries = [], [], []
matched = 0

for img_path in all_images:
    # We check if the Video ID (e.g., Abuse001_x264) is inside the path string
    # This covers both folder names and filenames.
    found_id = None
    for vid_id in video_lookup.keys():
        if vid_id in img_path:
            found_id = vid_id
            break

    if found_id:
        matched += 1
        description = random.choice(video_lookup[found_id])

        entry = {
            "id": os.path.basename(img_path),
            "image": img_path,
            "conversations": [
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Describe the events in this surveillance frame."}]},
                {"role": "assistant", "content": [{"type": "text", "text": description}]}
            ]
        }

        # Assign to correct split
        if found_id in train_json:
            train_entries.append(entry)
        elif found_id in val_json:
            val_entries.append(entry)
        else:
            test_entries.append(entry)

# 3. SAVE
def save_split(data, name):
    path = f"{MANIFEST_DIR}/{name}.jsonl"
    with jsonlines.open(path, 'w') as w: w.write_all(data)
    print(f"[RESULT] {name}: {len(data)} samples.")

save_split(train_entries, "train")
save_split(val_entries, "val")
save_split(test_entries, "test")

print("-" * 30)
print(f"[SUMMARY] Successfully matched {matched} out of {len(all_images)} images.")

[STATUS] Loading JSON Data...
[INFO] Loaded descriptions for 1854 videos.
[STATUS] Scanning Images and Matching...
[RESULT] train: 400 samples.
[RESULT] val: 100 samples.
[RESULT] test: 0 samples.
------------------------------
[SUMMARY] Successfully matched 500 out of 500 images.


In [ ]:
"""
STEP 2: Load New Data & Start Long-Haul Training
================================================
1. CONNECTS to the fresh manifests (train.jsonl / val.jsonl).
2. CONFIGURES the 260-Epoch Run.
3. SAVES to a new clean folder 'FireLLaVA_Real_v1'.
"""

from transformers import TrainingArguments, Trainer
from torch.utils.data import Dataset
import jsonlines
from PIL import Image
import os

# 1. DEFINE DATASET CLASS (To ensure we load the NEW manifest)
class FireLLaVADataset(Dataset):
    def __init__(self, manifest_path, processor):
        self.data = []
        if os.path.exists(manifest_path):
            with jsonlines.open(manifest_path) as reader:
                for obj in reader: self.data.append(obj)
        self.processor = processor

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item['image']).convert('RGB')

        # Format Text
        formatted_prompt = processor.apply_chat_template(
            item['conversations'],
            add_generation_prompt=False
        )

        # Process (Using 2048 Limit)
        inputs = self.processor(
            images=image,
            text=formatted_prompt,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=2048
        )

        return {
            "input_ids": inputs["input_ids"][0],
            "attention_mask": inputs["attention_mask"][0],
            "pixel_values": inputs["pixel_values"][0],
            "labels": inputs["input_ids"][0]
        }

# 2. LOAD THE NEW MANIFESTS
print("[STATUS] Loading new training data...")
train_dataset = FireLLaVADataset(f"{config.DRIVE_ROOT}/FireLLaVA_manifests/train.jsonl", processor)
val_dataset = FireLLaVADataset(f"{config.DRIVE_ROOT}/FireLLaVA_manifests/val.jsonl", processor)

print(f"[INFO] Train Size: {len(train_dataset)}")
print(f"[INFO] Val Size: {len(val_dataset)}")

# 3. CONFIGURE THE LONG HAUL
NEW_OUTPUT_DIR = f"{config.DRIVE_ROOT}/FireLLaVA_Real_v1"
os.makedirs(NEW_OUTPUT_DIR, exist_ok=True)

args = TrainingArguments(
    output_dir=NEW_OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=260,              # 260 Epochs
    learning_rate=2e-4,
    fp16=True,

    # Save & Eval Strategy
    eval_strategy="epoch",             # Check val loss every epoch
    save_strategy="epoch",             # Save model every epoch
    save_total_limit=1,                # Keep only the LATEST checkpoint
    load_best_model_at_end=True,       # Auto-load the best model at the end
    metric_for_best_model="eval_loss",

    logging_steps=10,
    report_to="none",
    remove_unused_columns=False
)

# 4. START TRAINER
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print(f"[STATUS] Starting Training in: {NEW_OUTPUT_DIR}")
print("Monitor the 'Eval Loss' column. Target is ~0.8.")

trainer.train()

print("[STATUS] Saving Final Best Model...")
trainer.save_model(f"{NEW_OUTPUT_DIR}/final_real_model")
print("[SUCCESS] Long Haul Complete.")

[STATUS] Loading new training data...
[INFO] Train Size: 400
[INFO] Val Size: 100
[STATUS] Starting Training in: /content/drive/MyDrive/ScenesolverG338LLaVAcoreLLM/FireLLaVA_Real_v1
Monitor the 'Eval Loss' column. Target is ~0.8.


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 688.00 MiB. GPU 0 has a total capacity of 22.16 GiB of which 363.38 MiB is free. Process 13042 has 21.80 GiB memory in use. Of the allocated memory 15.66 GiB is allocated by PyTorch, and 5.90 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
"""
Fire-LLaVA: A100 HIGH-SPEC TRAINING
===================================
1. HARDWARE: Optimized for NVIDIA A100 (40GB+ VRAM).
2. TOKENS: Restored to 2048 (No Compromise).
3. OPTIMIZER: Paged 8-bit AdamW (Maximum Stability).
4. RESUME: Auto-detects previous checkpoints.
"""

# Install dependencies
import os
print("[STATUS] Installing dependencies...")
os.system('pip install -q -U transformers accelerate peft bitsandbytes jsonlines')

import glob
import torch
import time
import jsonlines
import warnings
import gc
from PIL import Image
from google.colab import drive, runtime
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig,
    TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Memory Fragmentation Fix
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Force cleanup
gc.collect()
torch.cuda.empty_cache()

# 1. SETUP & PATHS
print("[STATUS] Setting up environment...")
warnings.filterwarnings('ignore')
drive.mount('/content/drive', force_remount=True)

class Config:
    DRIVE_ROOT = "/content/drive/MyDrive/ScenesolverG338LLaVAcoreLLM"
    OUTPUT_DIR = f"{DRIVE_ROOT}/FireLLaVA_Real_v1"
    MANIFEST_DIR = f"{DRIVE_ROOT}/FireLLaVA_manifests"
    MODEL_ID = "llava-hf/llava-1.5-7b-hf"

    # [RESTORED] No Compromise.
    MAX_LENGTH = 2048

config = Config()

# 2. RE-LOAD MODEL & PROCESSOR
print("[STATUS] Re-loading Model and Processor...")

processor = AutoProcessor.from_pretrained(config.MODEL_ID)
processor.patch_size = 14
processor.vision_feature_select_strategy = "default"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)

model = LlavaForConditionalGeneration.from_pretrained(
    config.MODEL_ID, quantization_config=bnb_config, device_map="auto"
)

# 3. APPLY OPTIMIZATIONS
print("[STATUS] Enabling Gradient Checkpointing...")
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# Re-attach LoRA
lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()

# 4. DATASET CLASS
class FireLLaVADataset(Dataset):
    def __init__(self, manifest_path, processor):
        self.data = []
        if os.path.exists(manifest_path):
            with jsonlines.open(manifest_path) as reader:
                for obj in reader: self.data.append(obj)
        self.processor = processor

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item['image']).convert('RGB')
        formatted_prompt = processor.apply_chat_template(
            item['conversations'], add_generation_prompt=False
        )
        inputs = self.processor(
            images=image, text=formatted_prompt, return_tensors="pt",
            padding="max_length", truncation=True, max_length=config.MAX_LENGTH
        )
        return {
            "input_ids": inputs["input_ids"][0],
            "attention_mask": inputs["attention_mask"][0],
            "pixel_values": inputs["pixel_values"][0],
            "labels": inputs["input_ids"][0]
        }

print("[STATUS] Loading Datasets...")
train_dataset = FireLLaVADataset(f"{config.MANIFEST_DIR}/train.jsonl", processor)
val_dataset = FireLLaVADataset(f"{config.MANIFEST_DIR}/val.jsonl", processor)

# 5. CONFIGURE TRAINER
args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=260,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,

    # We keep Paged Optimizer because it is safer, even on A100.
    optim="paged_adamw_8bit",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# 6. RESUME LOGIC
checkpoints = glob.glob(f"{config.OUTPUT_DIR}/checkpoint-*")
resume_flag = False

if len(checkpoints) > 0:
    print(f"[STATUS] Found {len(checkpoints)} existing checkpoint(s).")
    latest_checkpoint = max(checkpoints, key=os.path.getmtime)
    print(f"[STATUS] Resuming from latest: {os.path.basename(latest_checkpoint)}")
    resume_flag = True
else:
    print("[STATUS] Starting fresh training on A100.")
    resume_flag = False

print("[STATUS] Starting High-Spec A100 Training Run...")
trainer.train(resume_from_checkpoint=resume_flag)

print("[STATUS] Saving Final Model...")
trainer.save_model(f"{config.OUTPUT_DIR}/final_real_model")

# 7. DISCONNECT
print("[STATUS] Training Complete. Waiting 60s for Drive sync...")
time.sleep(60)
print("[STATUS] Disconnecting Runtime.")
runtime.unassign()

[STATUS] Installing dependencies...
[STATUS] Setting up environment...
Mounted at /content/drive
[STATUS] Re-loading Model and Processor...


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

[STATUS] Enabling Gradient Checkpointing...
[STATUS] Loading Datasets...
[STATUS] Starting fresh training on A100.
[STATUS] Starting High-Spec A100 Training Run...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,9.688600,9.642143
2,9.591400,9.549836
3,9.563400,9.536276
4,9.550300,9.537518
5,9.553200,9.539665
6,9.548600,9.544917
7,9.544200,9.548409
8,9.540300,9.558418
9,9.548300,9.557742
10,9.551800,9.562240


Epoch,Training Loss,Validation Loss
1,9.688600,9.642143
2,9.591400,9.549836
3,9.563400,9.536276
4,9.550300,9.537518
5,9.553200,9.539665
6,9.548600,9.544917
7,9.544200,9.548409
8,9.540300,9.558418
9,9.548300,9.557742
10,9.551800,9.562240


KeyboardInterrupt: 

In [ ]:
"""
Fire-LLaVA: AGGRESSIVE TRAINING (The Lesson)
============================================
1. LEARNING RATE: 5e-4 (2.5x higher).
2. LORA RANK: 32 (2x capacity).
3. LORA ALPHA: 64 (Stronger influence).
4. GOAL: Force the loss below 9.0 immediately.
"""

# Install dependencies
import os
print("[STATUS] Installing dependencies...")
os.system('pip install -q -U transformers accelerate peft bitsandbytes jsonlines')

import glob
import torch
import time
import jsonlines
import warnings
import gc
from PIL import Image
from google.colab import drive, runtime
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig,
    TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Memory Fix
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

# 1. SETUP
print("[STATUS] Setting up environment...")
warnings.filterwarnings('ignore')
drive.mount('/content/drive', force_remount=True)

class Config:
    DRIVE_ROOT = "/content/drive/MyDrive/ScenesolverG338LLaVAcoreLLM"
    # [NOTE] New folder for the aggressive run
    OUTPUT_DIR = f"{DRIVE_ROOT}/FireLLaVA_Aggressive_v1"
    MANIFEST_DIR = f"{DRIVE_ROOT}/FireLLaVA_manifests"
    MODEL_ID = "llava-hf/llava-1.5-7b-hf"
    MAX_LENGTH = 2048

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

# 2. MODEL & PROCESSOR
print("[STATUS] Re-loading Model...")
processor = AutoProcessor.from_pretrained(config.MODEL_ID)
processor.patch_size = 14
processor.vision_feature_select_strategy = "default"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)

model = LlavaForConditionalGeneration.from_pretrained(
    config.MODEL_ID, quantization_config=bnb_config, device_map="auto"
)

# 3. OPTIMIZATIONS
print("[STATUS] Enabling Gradient Checkpointing...")
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# [CHANGE] Aggressive LoRA Configuration
print("[STATUS] Applying AGGRESSIVE LoRA Config (Rank 32, Alpha 64)...")
lora_config = LoraConfig(
    r=32,                # Doubled Capacity
    lora_alpha=64,       # Doubled Influence
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()

# 4. DATASET
class FireLLaVADataset(Dataset):
    def __init__(self, manifest_path, processor):
        self.data = []
        if os.path.exists(manifest_path):
            with jsonlines.open(manifest_path) as reader:
                for obj in reader: self.data.append(obj)
        self.processor = processor

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item['image']).convert('RGB')
        formatted_prompt = processor.apply_chat_template(
            item['conversations'], add_generation_prompt=False
        )
        inputs = self.processor(
            images=image, text=formatted_prompt, return_tensors="pt",
            padding="max_length", truncation=True, max_length=config.MAX_LENGTH
        )
        return {
            "input_ids": inputs["input_ids"][0],
            "attention_mask": inputs["attention_mask"][0],
            "pixel_values": inputs["pixel_values"][0],
            "labels": inputs["input_ids"][0]
        }

print("[STATUS] Loading Datasets...")
train_dataset = FireLLaVADataset(f"{config.MANIFEST_DIR}/train.jsonl", processor)
val_dataset = FireLLaVADataset(f"{config.MANIFEST_DIR}/val.jsonl", processor)

# 5. TRAINER (Aggressive Settings)
args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=260,

    # [CHANGE] Higher Learning Rate
    learning_rate=5e-4,

    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# ==========================================
# UNIVERSAL START/RESUME LOGIC
# ==========================================
# Check if we have any existing checkpoints in the output folder
checkpoints = glob.glob(f"{config.OUTPUT_DIR}/checkpoint-*")

if len(checkpoints) > 0:
    # If we found files, we RESUME from the latest one
    latest_checkpoint = max(checkpoints, key=os.path.getmtime)
    print(f"[STATUS] Resume Needed. Found checkpoint: {os.path.basename(latest_checkpoint)}")
    print("[STATUS] Resuming Training...")
    trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    # If folder is empty, we START FRESH
    print("[STATUS] No previous checkpoints found. Starting FRESH Aggressive Run.")
    trainer.train()
# ==========================================

print("[STATUS] Saving Final Model...")
trainer.save_model(f"{config.OUTPUT_DIR}/final_real_model")

# 6. DISCONNECT
print("[STATUS] Training Complete. Waiting 60s for Drive sync...")
time.sleep(60)
print("[STATUS] Disconnecting Runtime.")
runtime.unassign()

[STATUS] Installing dependencies...
[STATUS] Setting up environment...
Mounted at /content/drive
[STATUS] Re-loading Model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[STATUS] Enabling Gradient Checkpointing...
[STATUS] Applying AGGRESSIVE LoRA Config (Rank 32, Alpha 64)...
trainable params: 19,922,944 || all params: 7,083,350,016 || trainable%: 0.2813
[STATUS] Loading Datasets...
[STATUS] Starting AGGRESSIVE Training Run...


Epoch,Training Loss,Validation Loss
1,9.575400,9.542290
2,9.560900,9.533978
3,9.546300,9.539979
4,9.534000,9.541827
5,9.542500,9.548587
6,9.540400,9.552847
7,9.540300,9.551807
8,9.537200,9.556329
9,9.546100,9.558067
10,9.550500,9.558918


KeyboardInterrupt: 

In [ ]:
"""
Fire-LLaVA: REFINEMENT RUN (Shift Gears)
========================================
1. LOADS: Weights from the latest 'Vision' checkpoint.
2. CHANGES: Learning Rate dropped to 2e-5 (Precision Mode).
3. GOAL: Stop oscillating and descend to < 1.0.
"""

import os
import glob
import torch
import warnings
import gc
from PIL import Image
from google.colab import drive, runtime
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig,
    TrainingArguments, Trainer
)
from peft import PeftModel, prepare_model_for_kbit_training

# Memory Fixes
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

# 1. SETUP
warnings.filterwarnings('ignore')
drive.mount('/content/drive', force_remount=True)

class Config:
    DRIVE_ROOT = "/content/drive/MyDrive/ScenesolverG338LLaVAcoreLLM"
    # Source Folder (Where we take weights from)
    SOURCE_DIR = f"{DRIVE_ROOT}/FireLLaVA_Vision_v1"
    # Destination Folder (Where we save new refined weights)
    OUTPUT_DIR = f"{DRIVE_ROOT}/FireLLaVA_Refine_v1"
    MANIFEST_DIR = f"{DRIVE_ROOT}/FireLLaVA_manifests"
    MODEL_ID = "llava-hf/llava-1.5-7b-hf"
    MAX_LENGTH = 2048

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

# 2. FIND LATEST CHECKPOINT
print("[STATUS] Locating previous checkpoint...")
checkpoints = glob.glob(f"{config.SOURCE_DIR}/checkpoint-*")
if len(checkpoints) == 0:
    raise ValueError("No checkpoints found in Vision folder! Cannot refine.")

# Pick the latest one
latest_checkpoint = max(checkpoints, key=os.path.getmtime)
print(f"[INFO] Loading weights from: {os.path.basename(latest_checkpoint)}")

# 3. LOAD BASE MODEL
print("[STATUS] Loading Base Model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
model = LlavaForConditionalGeneration.from_pretrained(
    config.MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
processor = AutoProcessor.from_pretrained(config.MODEL_ID)
processor.patch_size = 14
processor.vision_feature_select_strategy = "default"

# 4. LOAD TRAINED ADAPTERS (The "Resume" Logic)
print("[STATUS] Merging Trained Weights...")
model = prepare_model_for_kbit_training(model)

# Load the LoRA weights from the checkpoint
model = PeftModel.from_pretrained(model, latest_checkpoint, is_trainable=True)

# Ensure gradients are enabled for training
for name, param in model.named_parameters():
    if "lora" in name:
        param.requires_grad = True

model.enable_input_require_grads()
model.gradient_checkpointing_enable()

print("[STATUS] Model successfully loaded. Ready for Refinement.")

# 5. DATASET
class FireLLaVADataset(Dataset):
    def __init__(self, manifest_path, processor):
        self.data = []
        if os.path.exists(manifest_path):
            with jsonlines.open(manifest_path) as reader:
                for obj in reader: self.data.append(obj)
        self.processor = processor

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item['image']).convert('RGB')
        formatted_prompt = processor.apply_chat_template(
            item['conversations'], add_generation_prompt=False
        )
        inputs = self.processor(
            images=image, text=formatted_prompt, return_tensors="pt",
            padding="max_length", truncation=True, max_length=config.MAX_LENGTH
        )
        return {
            "input_ids": inputs["input_ids"][0],
            "attention_mask": inputs["attention_mask"][0],
            "pixel_values": inputs["pixel_values"][0],
            "labels": inputs["input_ids"][0]
        }

print("[STATUS] Loading Datasets...")
train_dataset = FireLLaVADataset(f"{config.MANIFEST_DIR}/train.jsonl", processor)
val_dataset = FireLLaVADataset(f"{config.MANIFEST_DIR}/val.jsonl", processor)

# 6. TRAINER (Low Learning Rate)
args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=50,  # Refinement doesn't need 260 epochs usually

    # [CRITICAL CHANGE] Low Learning Rate for Precision
    learning_rate=2e-5,

    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("[STATUS] Starting REFINEMENT Run (LR: 2e-5)...")
trainer.train()

print("[STATUS] Saving Final Refined Model...")
trainer.save_model(f"{config.OUTPUT_DIR}/final_refined_model")

# 7. DISCONNECT
print("[STATUS] Training Complete. Waiting 60s for Drive sync...")
time.sleep(60)
print("[STATUS] Disconnecting Runtime.")
runtime.unassign()

Mounted at /content/drive
[STATUS] Locating previous checkpoint...
[INFO] Loading weights from: checkpoint-400
[STATUS] Loading Base Model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[STATUS] Merging Trained Weights...
[STATUS] Model successfully loaded. Ready for Refinement.
[STATUS] Loading Datasets...
[STATUS] Starting REFINEMENT Run (LR: 2e-5)...


Epoch,Training Loss,Validation Loss
1,3.760600,3.822650
2,3.764500,3.825397
3,3.762600,3.830010


KeyboardInterrupt: 

In [6]:
"""
Fire-LLaVA: PHASE 2 - DEEP VISION (Surveillance Upgrade)
======================================================
FIXED:
- Chat template error resolved (conversations format fixed)
- Processor crash resolved
- Vision + LLM + Bridge LoRA preserved
"""

import os
import gc
import warnings

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ---------------------------------------------------
# 1. INSTALL DEPENDENCIES
# ---------------------------------------------------
print("[STATUS] Installing stable dependencies...")
os.system("pip install -q -U bitsandbytes")
os.system("pip install -q -U transformers==4.37.2 peft==0.8.2 accelerate==0.27.2 jsonlines")

# ---------------------------------------------------
# 2. IMPORTS
# ---------------------------------------------------
import torch
from PIL import Image
from google.colab import drive, runtime
from torch.utils.data import Dataset

from transformers import (
    LlavaForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    AutoTokenizer,
    LlavaProcessor,
    CLIPImageProcessor
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

# ---------------------------------------------------
# 3. MEMORY CLEANUP
# ---------------------------------------------------
warnings.filterwarnings("ignore")
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------------
# 4. DRIVE MOUNT
# ---------------------------------------------------
drive.mount("/content/drive", force_remount=True)

# ---------------------------------------------------
# 5. CONFIG
# ---------------------------------------------------
class Config:
    DRIVE_ROOT = "/content/drive/MyDrive/ScenesolverG338LLaVAcoreLLM"
    OUTPUT_DIR = f"{DRIVE_ROOT}/FireLLaVA_Deep_v1"
    MANIFEST_DIR = f"{DRIVE_ROOT}/FireLLaVA_manifests"
    MODEL_ID = "llava-hf/llava-1.5-7b-hf"
    MAX_LENGTH = 2048

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------
# 6. LOAD MODEL
# ---------------------------------------------------
print("[STATUS] Loading base model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = LlavaForConditionalGeneration.from_pretrained(
    config.MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# ---------------------------------------------------
# 7. SAFE PROCESSOR LOAD
# ---------------------------------------------------
print("[STATUS] Loading processor (hard-fix mode)...")

tokenizer = AutoTokenizer.from_pretrained(
    config.MODEL_ID,
    use_fast=False
)

image_processor = CLIPImageProcessor.from_pretrained(config.MODEL_ID)

processor = LlavaProcessor(
    tokenizer=tokenizer,
    image_processor=image_processor
)

processor.patch_size = 14
processor.vision_feature_select_strategy = "default"

print("[STATUS] Processor loaded safely.")

# ---------------------------------------------------
# 8. PREPARE MODEL + DEEP VISION LORA
# ---------------------------------------------------
print("[STATUS] Applying Deep Vision LoRA...")

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        # LLM
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",

        # Vision Tower
        "vision_tower.vision_model.encoder.layers.*.self_attn.q_proj",
        "vision_tower.vision_model.encoder.layers.*.self_attn.k_proj",
        "vision_tower.vision_model.encoder.layers.*.self_attn.v_proj",
        "vision_tower.vision_model.encoder.layers.*.self_attn.out_proj",

        # Bridge
        "mm_projector"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ---------------------------------------------------
# 9. DATASET (FIXED - Simple prompt format)
# ---------------------------------------------------
class FireLLaVADataset(Dataset):
    def __init__(self, manifest_path, processor):
        import jsonlines
        self.processor = processor
        self.data = []

        if os.path.exists(manifest_path):
            with jsonlines.open(manifest_path) as reader:
                for obj in reader:
                    self.data.append(obj)

        print(f"Loaded {len(self.data)} samples from {manifest_path}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Load image
        try:
            image = Image.open(item["image"]).convert("RGB")
        except Exception as e:
            print(f"Error loading image {item['image']}: {e}")
            # Return next item as fallback
            return self.__getitem__((idx + 1) % len(self.data))

        # FIXED: Simple prompt format without chat template
        # Extract text from conversations if it exists
        if "conversations" in item:
            # Handle list format: [{"from": "human", "value": "..."}, {"from": "gpt", "value": "..."}]
            if isinstance(item["conversations"], list):
                user_msg = ""
                assistant_msg = ""
                for conv in item["conversations"]:
                    if conv.get("from") == "human" or conv.get("role") == "user":
                        user_msg = conv.get("value", "")
                    elif conv.get("from") == "gpt" or conv.get("role") == "assistant":
                        assistant_msg = conv.get("value", "")

                # Create simple prompt
                prompt = f"USER: <image>\n{user_msg}\nASSISTANT: {assistant_msg}"
            else:
                # Fallback to caption if conversations is not a list
                prompt = f"USER: <image>\nDescribe this image.\nASSISTANT: {item.get('caption', 'A surveillance scene.')}"
        else:
            # Use caption field if no conversations
            caption = item.get("caption", "A surveillance scene.")
            prompt = f"USER: <image>\nDescribe this surveillance video clip.\nASSISTANT: {caption}"

        try:
            # Process with processor
            inputs = self.processor(
                images=image,
                text=prompt,
                return_tensors="pt",
                padding="max_length",
                truncation=True,
                max_length=config.MAX_LENGTH
            )

            # Return properly formatted batch
            return {
                "input_ids": inputs["input_ids"].squeeze(0),
                "attention_mask": inputs["attention_mask"].squeeze(0),
                "pixel_values": inputs["pixel_values"].squeeze(0),
                "labels": inputs["input_ids"].squeeze(0)
            }
        except Exception as e:
            print(f"Error processing item {idx}: {e}")
            # Return next item as fallback
            return self.__getitem__((idx + 1) % len(self.data))

print("[STATUS] Loading datasets...")

train_dataset = FireLLaVADataset(
    f"{config.MANIFEST_DIR}/train.jsonl",
    processor
)

val_dataset = FireLLaVADataset(
    f"{config.MANIFEST_DIR}/val.jsonl",
    processor
)

# Check if datasets loaded successfully
if len(train_dataset) == 0:
    raise ValueError(f"No training data found in {config.MANIFEST_DIR}/train.jsonl")
if len(val_dataset) == 0:
    print(f"Warning: No validation data found in {config.MANIFEST_DIR}/val.jsonl")
    val_dataset = None

# ---------------------------------------------------
# 10. TRAINING
# ---------------------------------------------------
args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=15,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    evaluation_strategy="steps" if val_dataset else "no",
    eval_steps=20 if val_dataset else None,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("[STATUS] Starting Deep Vision training...")
print(f"Training samples: {len(train_dataset)}")
if val_dataset:
    print(f"Validation samples: {len(val_dataset)}")

trainer.train()

# ---------------------------------------------------
# 11. SAVE
# ---------------------------------------------------
print("[STATUS] Saving final model...")
trainer.save_model(f"{config.OUTPUT_DIR}/final_deep_model")
processor.save_pretrained(f"{config.OUTPUT_DIR}/final_deep_model")

print("[STATUS] Model and processor saved successfully!")

# ---------------------------------------------------
# 12. CLEAN EXIT
# ---------------------------------------------------
print("[STATUS] Syncing Drive...")
import time
time.sleep(30)

print("[STATUS] Training complete!")
print(f"Model saved to: {config.OUTPUT_DIR}/final_deep_model")

[STATUS] Installing stable dependencies...
Mounted at /content/drive
[STATUS] Loading base model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


[STATUS] Loading processor (hard-fix mode)...
[STATUS] Processor loaded safely.
[STATUS] Applying Deep Vision LoRA...
trainable params: 84,672,512 || all params: 7,148,099,584 || trainable%: 1.1845457803851436
[STATUS] Loading datasets...
Loaded 400 samples from /content/drive/MyDrive/ScenesolverG338LLaVAcoreLLM/FireLLaVA_manifests/train.jsonl
Loaded 100 samples from /content/drive/MyDrive/ScenesolverG338LLaVAcoreLLM/FireLLaVA_manifests/val.jsonl
[STATUS] Starting Deep Vision training...
Training samples: 400
Validation samples: 100


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 882.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 620.12 MiB is free. Process 21720 has 14.13 GiB memory in use. Of the allocated memory 13.66 GiB is allocated by PyTorch, and 343.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)